# Sentiment Analysis

### Importing Libraries

In [67]:
import pandas as pd
import numpy as np
import spacy
from sklearn.model_selection import train_test_split
import re
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from sklearn.preprocessing import LabelEncoder
import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from sklearn.metrics import confusion_matrix, classification_report


### Importing Data & Splitting 

In [68]:
data = pd.read_csv('../Data/sentiment_analysis.csv', index_col= False)
nlp = spacy.load("en_core_web_sm")

y = data['sentiment']
X = data.drop(columns= ['sentiment'])

train, temp, y_train, y_temp = train_test_split(
    X, y, test_size= 0.3, random_state= 42, stratify= y
)

test, validation, y_test, y_validation = train_test_split(
    temp, y_temp, test_size= 0.5, random_state= 42, stratify= y_temp
)

### Pre-Processing Data

In [69]:
# lowercased the whole text
def lowercase(df):
    df = df.copy()
    df['text'] = df['text'].str.lower()
    return df

# removed Urls from the text
def remove_urls(df):
    df = df.copy()
    df['text'] = df['text'].apply(lambda x: re.sub(r'https\S+|www\S+', '', x))
    return df

# removed numbers from the text
def remove_numbers(df):
    df = df.copy()
    df['text'] = df['text'].apply(lambda x: re.sub(r'\d+', '', x))
    return df

# handling stopwords with removing negation terms
negations = {'not', 'no', 'nor', 'never', "n't", "cannot", "none", "neither"}
stopwords = nlp.Defaults.stop_words - negations

def spacy_clean(df):
    df = df.copy()
    df['text'] = df['text'].apply(
        lambda x: ' '. join([
            t.lemma_ for t in nlp(x)
            if not t.is_punct and t.text not in stopwords
        ])
    )
    return df

# the above function do the work of all three functions 
# def remove_stopwords(df):
#     df = df.copy()
#     df['text'] = df['text'].apply(lambda x: ' '.join([t.text for t in nlp(x) if t.text not in stopwords]))
#     return df
# # lemmatize 
# def lemmatize(df):
#     df = df.copy()
#     df['text'] = df['text'].apply(lambda x: ' '.join([t.lemma_ for t in nlp(x)]))
#     return df
# # punctuation removal 
# def remove_punctuation(df):
#     df = df.copy()
#     df['text'] = df['text'].apply(lambda x: ' '.join([t.text for t in nlp(x) if not t.is_punct]))
#     return df

# As I'm using bert model then I don't need these steps so I am just removing them from from clean text.

def clean_text(df):
    df = df.copy()
    df = lowercase(df)
    df = remove_urls(df)
    # df = remove_numbers(df)
    # df = spacy_clean(df)
    return df

train = clean_text(train)
test = clean_text(test)
validation = clean_text(validation)

### Tokenizer with BERT

In [70]:
tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')

def token(df):
    return tokenizer(
        list(df['text']),
        padding = True,
        truncation = True,
        max_length = 256,
        return_tensors = 'pt'
    )

train = token(train)
test = token(test)
validation = token(validation)

### Label Encoding the output

In [71]:
encoder = LabelEncoder()

y_train = encoder.fit_transform(y_train)
y_test = encoder.transform(y_test)
y_validation = encoder.transform(y_validation)

### Building Pytorch Dataset object Pair

In [72]:
class Sentiment(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, index):
        item = {key: val[index] for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[index])
        return item

train_dataset = Sentiment(train, y_train)
val_dataset = Sentiment(validation, y_validation)
test_dataset = Sentiment(test, y_test)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)

### Loading Model

In [73]:
num_classes = len(encoder.classes_)

model = AutoModelForSequenceClassification.from_pretrained(
    'bert-base-uncased',
    num_labels = num_classes
)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2624.92it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoi

BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,),

### Training the model

In [74]:
training_args = TrainingArguments(
    output_dir='./results',
    save_strategy='no',
    num_train_epochs=10,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=2e-5,
    weight_decay=0.01,
    eval_strategy='epoch',
    logging_steps=10,
    load_best_model_at_end=False,
    metric_for_best_model='accuracy'
)

### Accuracy 

In [75]:
def compute_metrics(pred):
    labels = pred.label_ids
    preds = np.argmax(pred.predictions, axis=1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='weighted')
    acc = accuracy_score(labels, preds)
    return {
        'accuracy': acc,
        'f1': f1,
        'precision': precision,
        'recall': recall
    }

In [76]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics
)

In [77]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,1.081530,1.041016,0.506667,0.418917,0.437055,0.506667
2,0.959600,0.836029,0.666667,0.602331,0.766667,0.666667
3,0.713155,0.676447,0.773333,0.766490,0.780918,0.773333
4,0.555808,0.586132,0.800000,0.793950,0.806636,0.800000
5,0.280848,0.505408,0.786667,0.781777,0.793804,0.786667
6,0.182135,0.538120,0.800000,0.793950,0.806636,0.800000
7,0.111111,0.526293,0.786667,0.778246,0.793383,0.786667
8,0.080361,0.576986,0.786667,0.778246,0.793383,0.786667
9,0.062301,0.543183,0.786667,0.781777,0.793804,0.786667
10,0.046060,0.550617,0.800000,0.793950,0.806636,0.800000


/home/harsh/Work/.venv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


TrainOutput(global_step=220, training_loss=0.41498274315487255, metrics={'train_runtime': 21.1028, 'train_samples_per_second': 165.381, 'train_steps_per_second': 10.425, 'total_flos': 87880909307940.0, 'train_loss': 0.41498274315487255, 'epoch': 10.0})

In [78]:
test_results = trainer.evaluate(test_dataset)
print(test_results)

Training Loss,Validation Loss,Epoch,Accuracy,F1,Precision,Recall
0.046060,0.942104,10,0.693333,0.691963,0.706553,0.693333


{'eval_loss': 0.9421035647392273, 'eval_accuracy': 0.6933333333333334, 'eval_f1': 0.6919625507860803, 'eval_precision': 0.7065531475748195, 'eval_recall': 0.6933333333333334}


In [79]:
predictions = trainer.predict(test_dataset)
preds = np.argmax(predictions.predictions, axis=1)
true_labels = predictions.label_ids

print(confusion_matrix(true_labels, preds))
print(classification_report(true_labels, preds, target_names=encoder.classes_))

[[13  5  2]
 [ 3 24  3]
 [ 1  9 15]]
              precision    recall  f1-score   support

    negative       0.76      0.65      0.70        20
     neutral       0.63      0.80      0.71        30
    positive       0.75      0.60      0.67        25

    accuracy                           0.69        75
   macro avg       0.72      0.68      0.69        75
weighted avg       0.71      0.69      0.69        75

